# Phase 1 — Data Access + Portfolio Agent

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** read the real Excel portfolios and answer holdings + performance questions.

**Data facts used here:** `CLT-002` holds ETFs + individual stocks + cash (ideal for the 3-type distinction). Cash `quantity` is a dollar balance; never multiply it by purchase price.


In [ ]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env')

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

## 1. Repository reads the real `.xlsx` (sheet 'Potfolios' — sic)

In [ ]:
from app.data.repositories import ExcelPortfolioRepository
repo = ExcelPortfolioRepository()
p = repo.get('CLT-002')
print('CLT-002 holdings:', len(p.holdings))
for h in p.holdings:
    print(f'  {h.symbol:6} {h.asset_class:22} qty={h.quantity} is_cash={h.is_cash} is_etf={h.is_etf}')

## 2. Cash is handled correctly (dollar balance, not qty x price)

In [ ]:
cash = [h for h in repo.get('CLT-001').holdings if h.is_cash][0]
print('CASH quantity (=$ balance):', cash.quantity)
print('CASH cost_basis          :', cash.cost_basis, '(should equal quantity, NOT quantity*price)')

## 2b. Allocation by asset class (stocks vs ETF types vs cash)

In [ ]:
from app.tools.portfolio_tools import get_allocation_by_asset_class, get_allocation_by_sector
print(get_allocation_by_asset_class('CLT-002'))
print(get_allocation_by_sector('CLT-002'))

## 3. Portfolio agent answers 'what do I own?' (grouped by asset class)

In [ ]:
from app.agents.portfolio import PortfolioAgent
agent = PortfolioAgent()
print(agent.answer(client_id='CLT-002', query='What do I own?'))

## 4. Performance tools (the brief's YTD + since-purchase questions)

In [ ]:
from app.tools.portfolio_tools import get_position_performance, get_ytd_returns
print('NVDA since purchase:', get_position_performance('CLT-002', 'NVDA'))
print('Best YTD return    :', get_ytd_returns('CLT-002'))

## ✅ Acceptance check

In [ ]:
ans = agent.answer(client_id='CLT-002', query='What do I own?').lower()
assert any(k in ans for k in ['etf', 'fund'])   # names an ETF
assert 'cash' in ans                            # names cash
print('Phase 1 acceptance: PASS')